In [1]:
import ollama

# Quick connectivity test
response = ollama.chat(model='llama3:latest', messages=[
    {'role': 'user', 'content': 'Say hello in one sentence.'}
])
print(response['message']['content'])

Hello there!


### Loading saved model + NLP artifacts

In [2]:
import pickle
import numpy as np

with open('medisense_preprocessed.pkl', 'rb') as f:
    data = pickle.load(f)

with open('nlp_artifacts.pkl', 'rb') as f:
    nlp_data = pickle.load(f)

xgb = data['xgb_model']
le = data['label_encoder']
feature_names = data['feature_names']
symptom_lookup = nlp_data['symptom_lookup']
symptom_phrases = nlp_data['symptom_phrases']
SYNONYMS = nlp_data['synonyms']

print("Model and NLP artifacts loaded ✓")

/home/sunbeam/.local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/sunbeam/.local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/sunbeam/.local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version

Model and NLP artifacts loaded ✓


In [8]:
import re
from fuzzywuzzy import fuzz, process

def generate_ngrams(text, max_n=5):
    words = text.split()
    ngrams = []
    for n in range(1, min(max_n, len(words)) + 1):
        for i in range(len(words) - n + 1):
            ngrams.append(' '.join(words[i:i+n]))
    return ngrams

def extract_symptoms_from_text(text, threshold=80):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    found_symptoms = set()
    working_text = text
    for syn, canonical in SYNONYMS.items():
        if syn in working_text:
            working_text += ' | ' + canonical

    sorted_phrases = sorted(symptom_phrases, key=len, reverse=True)
    for phrase in sorted_phrases:
        if phrase in working_text:
            found_symptoms.add(symptom_lookup[phrase])

    ngrams = [g for g in generate_ngrams(working_text, max_n=4) if len(g) >= 4 and '|' not in g]
    for gram in ngrams:
        match, score = process.extractOne(gram, symptom_phrases, scorer=fuzz.token_sort_ratio)
        if score >= threshold:
            found_symptoms.add(symptom_lookup[match])

    redundant_pairs = [('back pain', 'low back pain')]
    for general, specific in redundant_pairs:
        general_col = symptom_lookup.get(general)
        specific_col = symptom_lookup.get(specific)
        if general_col in found_symptoms and specific_col in found_symptoms:
            found_symptoms.discard(general_col)

    return list(found_symptoms)


# Symptoms too vague to be useful alone — still extracted for display, 
# but excluded from the feature vector unless paired with specific symptoms
GENERIC_SYMPTOMS = {'feeling ill', 'fatigue', 'weakness', 'ache all over'}

def predict_disease_from_text(text, model, feature_names, label_encoder, top_n=3):
    extracted = extract_symptoms_from_text(text)

    if not extracted:
        return {
            'extracted_symptoms': [],
            'predictions': [],
            'message': 'No recognizable symptoms found. Please describe your symptoms more specifically.'
        }

    # Use specific symptoms for prediction; only fall back to generic ones if nothing else exists
    specific = [s for s in extracted if s not in GENERIC_SYMPTOMS]
    symptoms_for_model = specific if specific else extracted

    feature_vector = np.zeros(len(feature_names))
    for symptom in symptoms_for_model:
        if symptom in feature_names:
            idx = feature_names.index(symptom)
            feature_vector[idx] = 1
    feature_vector = feature_vector.reshape(1, -1)

    probs = model.predict_proba(feature_vector)[0]
    top_indices = np.argsort(probs)[::-1][:top_n]

    predictions = [
        {
            'disease': label_encoder.inverse_transform([idx])[0],
            'confidence': round(float(probs[idx]) * 100, 2)
        }
        for idx in top_indices
    ]

    return {
        'extracted_symptoms': [s.replace('_', ' ') for s in extracted],  # show all, for transparency
        'symptoms_used_for_prediction': [s.replace('_', ' ') for s in symptoms_for_model],
        'predictions': predictions,
        'message': 'success'
    }

print("Functions loaded ✓")

Functions loaded ✓


### Report generator: turning prediction into plain-language summary

In [9]:
def generate_report(text, prediction_result):
    if not prediction_result['predictions']:
        return "I couldn't identify clear symptoms from your description. Could you describe what you're feeling in more detail?"

    symptoms_str = ", ".join(prediction_result['extracted_symptoms'])
    predictions_str = "\n".join([
        f"- {p['disease']}: {p['confidence']}% likelihood"
        for p in prediction_result['predictions']
    ])

    prompt = f"""You are a careful, friendly medical triage assistant. A patient described their symptoms as:
"{text}"

Symptoms identified: {symptoms_str}

The ML model's top predictions (NOT a confirmed diagnosis, just statistical likelihood based on symptom patterns):
{predictions_str}

Write a short, clear, reassuring summary for the patient covering:
1. What symptoms were understood
2. That these are POSSIBLE conditions to discuss with a doctor, not a diagnosis
3. A gentle recommendation to consult a healthcare professional, especially if symptoms are severe
4. Keep it under 120 words, warm but professional tone, no medical jargon

Do not state any of these as a confirmed diagnosis. Always emphasize this is a preliminary screening tool only."""

    response = ollama.chat(model='llama3.2:3b', messages=[
        {'role': 'user', 'content': prompt}
    ])
    return response['message']['content']


# Test
test_text = "I have joint pain and my hands feel stiff in the morning"
result = predict_disease_from_text(test_text, xgb, feature_names, le, top_n=3)
report = generate_report(test_text, result)
print(report)

"Thank you for sharing your symptoms with me. I understand that you're experiencing joint pain and stiffness in your hands, especially in the morning. These are common symptoms that can be associated with various conditions. Our top predictions suggest osteoporosis, sickle cell anemia, or chronic knee pain as possible possibilities to discuss further with a doctor. Please know that these are just probabilities based on patterns of symptoms. If you're experiencing severe or persistent joint pain, I strongly encourage you to consult with a healthcare professional for personalized guidance and care. We can provide you with more information and support to help you get the next steps."


### Clarifying-question chatbot (multi-turn)

In [10]:
def chat_with_patient(conversation_history, latest_user_message, turn_number, min_turns=3):
    # Only allow the model to suggest readiness after min_turns
    readiness_instruction = (
        "If the patient has now given enough detail (what hurts, how long, any other symptoms), "
        "end your response with READY_FOR_PREDICTION on its own line."
        if turn_number >= min_turns else
        "Do NOT say READY_FOR_PREDICTION yet — you still need more detail. "
        "Just ask one clarifying question."
    )

    system_prompt = f"""You are MediSense, a friendly medical triage chatbot.

Your job: gather symptom details through natural conversation before diagnosis.
Ask ONE short clarifying question at a time (duration, severity, location, associated symptoms).
Never diagnose yourself.
If symptoms sound urgent (severe chest pain, breathing difficulty, stroke signs), say so immediately.

{readiness_instruction}"""

    messages = [{'role': 'system', 'content': system_prompt}] + conversation_history + [
        {'role': 'user', 'content': latest_user_message}
    ]

    response = ollama.chat(model='llama3.2:3b', messages=messages)
    return response['message']['content']


def run_full_conversation_demo():
    history = []
    turns = [
        "I haven't been feeling well",
        "My joints hurt, especially in the morning",
        "It's been going on for about 2 weeks, mostly my hands and knees"
    ]

    for i, turn in enumerate(turns):
        print(f"\n👤 Patient: {turn}")
        reply = chat_with_patient(history, turn, turn_number=i+1, min_turns=3)
        # Strip the token from displayed text so it reads naturally
        display_reply = reply.replace("READY_FOR_PREDICTION", "").strip()
        print(f"🤖 MediSense: {display_reply}")

        history.append({'role': 'user', 'content': turn})
        history.append({'role': 'assistant', 'content': reply})

        is_last_turn = (i == len(turns) - 1)
        if "READY_FOR_PREDICTION" in reply or is_last_turn:
            full_text = " ".join([h['content'] for h in history if h['role'] == 'user'])
            result = predict_disease_from_text(full_text, xgb, feature_names, le, top_n=3)
            report = generate_report(full_text, result)
            print(f"\n📋 Final Report:\n{report}")
            break

run_full_conversation_demo()


👤 Patient: I haven't been feeling well
🤖 MediSense: Can you tell me where in your body you're experiencing discomfort or symptoms?

👤 Patient: My joints hurt, especially in the morning
🤖 MediSense: Do the joint pains get better or worse as the day goes on, and are there any activities that seem to make them feel particularly worse?

👤 Patient: It's been going on for about 2 weeks, mostly my hands and knees
🤖 MediSense: Have you noticed any other symptoms, such as fever, fatigue, swollen lymph nodes, or difficulty moving your joints?

📋 Final Report:
"Thank you for sharing your concerns with me. I've understood that you're experiencing joint pain, especially in the morning, and feeling unwell overall. These symptoms have been ongoing for about 2 weeks, mainly affecting your hands and knees. Our model suggests a few possible conditions we can discuss with a doctor: osteoporosis, sickle cell anemia, and chronic knee pain. Please know that these are just possibilities, not a definitive di

###  Multilingual support (Hindi/Marathi)

In [11]:
def generate_report_multilingual(text, prediction_result, language='English'):
    if not prediction_result['predictions']:
        return "I couldn't identify clear symptoms. Could you describe what you're feeling in more detail?"

    symptoms_str = ", ".join(prediction_result['extracted_symptoms'])
    predictions_str = "\n".join([
        f"- {p['disease']}: {p['confidence']}% likelihood"
        for p in prediction_result['predictions']
    ])

    prompt = f"""You are a medical triage assistant. Respond ONLY in {language}.

Patient described: "{text}"
Symptoms identified: {symptoms_str}
Possible conditions (not a diagnosis):
{predictions_str}

Write a short, warm summary in {language} (under 120 words) explaining:
1. What symptoms were understood
2. These are possibilities to discuss with a doctor, not a diagnosis
3. Recommend seeing a healthcare professional
Respond entirely in {language} using appropriate script (Devanagari for Hindi/Marathi)."""

    response = ollama.chat(model='llama3.2:3b', messages=[
        {'role': 'user', 'content': prompt}
    ])
    return response['message']['content']


# Test in Hindi
test_text = "I have joint pain and my hands feel stiff in the morning"
result = predict_disease_from_text(test_text, xgb, feature_names, le, top_n=3)

print("=== English ===")
print(generate_report_multilingual(test_text, result, 'English'))

print("\n=== Hindi ===")
print(generate_report_multilingual(test_text, result, 'Hindi'))

=== English ===
"Hello! I understand that you're experiencing joint pain and stiffness in your hands, especially in the morning. These symptoms are concerning and I'd like to discuss some possible causes with our doctor.

The conditions we've identified as possibilities are osteoporosis, sickle cell anemia, and chronic knee pain. Please note that these are just potential explanations for your symptoms and not a definitive diagnosis.

I strongly recommend that you schedule an appointment with our healthcare professional to further evaluate your condition. They will be able to assess your symptoms, perform any necessary tests, and provide a proper diagnosis and treatment plan."

=== Hindi ===
"प्रिय रोगी, मैं आपके लक्षणों को समझना चाहता हूँ। आपके हाथों में सामान्य दर्द और सुबह की शुरुआत में थकान महसूस होना।
इन लक्षणों को एक डॉक्टर के साथ विचार करना आवश्यक है। ये संभावनाएं हैं, लेकिन यह एक निश्चित रोग नहीं है।
मैं आपको सलाह देता हूँ कि तुरंत एक स्वास्थ्य पेशेवर से मिलना चाहिए। वे आपके लक्

In [10]:
full_text = "I haven't been feeling well My joints hurt, especially in the morning It's been going on for about 2 weeks, mostly my hands and knees"
extracted = extract_symptoms_from_text(full_text)
print("Extracted symptoms:", extracted)

Extracted symptoms: ['feeling ill']


In [11]:
SYNONYMS.update({
    'joints hurt': 'joint pain',
    'joint hurts': 'joint pain',
    'my joints': 'joint pain',  # broader catch, use carefully
    'hands hurt': 'hand or finger pain',
    'knees hurt': 'knee pain',
    'knee hurts': 'knee pain',
})

extracted = extract_symptoms_from_text(full_text)
print("Extracted symptoms:", extracted)

Extracted symptoms: ['feeling ill', 'joint pain']


In [12]:
result = predict_disease_from_text(full_text, xgb, feature_names, le, top_n=3)
print(result)

{'extracted_symptoms': ['feeling ill', 'joint pain'], 'symptoms_used_for_prediction': ['joint pain'], 'predictions': [{'disease': 'osteoporosis', 'confidence': 59.27}, {'disease': 'sickle cell anemia', 'confidence': 11.75}, {'disease': 'chronic knee pain', 'confidence': 7.74}], 'message': 'success'}


In [14]:
def run_medisense_chat(user_inputs=None):
    """
    user_inputs: list of strings for demo mode.
    If None, runs interactively (for Streamlit later).
    """
    history = []
    turn = 0

    while True:
        # Get user input
        if user_inputs:
            if turn >= len(user_inputs):
                break
            user_msg = user_inputs[turn]
            print(f"\n👤 Patient: {user_msg}")
        else:
            user_msg = input("\n👤 You: ")
            if user_msg.lower() in ['quit', 'exit']:
                break

        # Build full text from conversation so far
        full_text_so_far = " ".join(
            [h['content'] for h in history if h['role'] == 'user']
        ) + " " + user_msg

        # Extract symptoms in real time
        symptoms_so_far = extract_symptoms_from_text(full_text_so_far)
        specific_symptoms = [s for s in symptoms_so_far if s not in GENERIC_SYMPTOMS]
        symptoms_str = ", ".join(s.replace('_', ' ') for s in specific_symptoms) \
                       if specific_symptoms else "none clearly identified yet"

        # Build system prompt with live symptom feedback
        turn += 1
        if turn >= 3 and len(specific_symptoms) >= 1:
            readiness_instruction = (
                f"Symptoms clearly identified so far: {symptoms_str}. "
                "Briefly confirm these back to the patient in plain language, "
                "then end your response with READY_FOR_PREDICTION on its own line."
            )
        elif turn >= 3:
            readiness_instruction = (
                f"Symptoms identified: {symptoms_str}. Not enough specific symptoms yet. "
                "Ask a more targeted question: location of pain, any fever/swelling/rash, "
                "or when symptoms started. Do NOT say READY_FOR_PREDICTION yet."
            )
        else:
            readiness_instruction = (
                f"Symptoms identified so far: {symptoms_str}. "
                "Ask ONE clarifying question. Do NOT say READY_FOR_PREDICTION yet."
            )

        system_prompt = f"""You are MediSense, a friendly medical triage chatbot.
Gather symptom details one question at a time. Never diagnose yourself.
For urgent symptoms (severe chest pain, difficulty breathing, stroke signs) advise emergency care immediately.
{readiness_instruction}"""

        messages = [{'role': 'system', 'content': system_prompt}] + history + [
            {'role': 'user', 'content': user_msg}
        ]
        response = ollama.chat(model='llama3.1:8b', messages=messages)
        reply = response['message']['content']

        display_reply = reply.replace("READY_FOR_PREDICTION", "").strip()
        print(f"🤖 MediSense: {display_reply}")

        history.append({'role': 'user', 'content': user_msg})
        history.append({'role': 'assistant', 'content': reply})

        # Trigger prediction
        if "READY_FOR_PREDICTION" in reply or (turn >= 3 and len(specific_symptoms) >= 1):
            result = predict_disease_from_text(full_text_so_far, xgb, feature_names, le, top_n=3)
            report = generate_report(full_text_so_far, result)
            print(f"\n{'='*55}")
            print(f"📋 MEDISENSE REPORT")
            print(f"{'='*55}")
            print(f"Symptoms identified : {', '.join(result['extracted_symptoms'])}")
            print(f"Used for prediction : {', '.join(result['symptoms_used_for_prediction'])}")
            print(f"\nTop predictions:")
            for p in result['predictions']:
                bar = '█' * int(p['confidence'] / 5)
                print(f"  {p['disease']:<40} {bar} {p['confidence']}%")
            print(f"\n💬 AI Summary:\n{report}")
            print(f"{'='*55}")
            print("\n⚠️  This is a screening tool only. Always consult a doctor.")
            break

# Demo run
run_medisense_chat(user_inputs=[
    "I haven't been feeling well",
    "My joints hurt, especially in the morning",
    "It's been going on for about 2 weeks, mostly my hands and knees"
])


👤 Patient: I haven't been feeling well
🤖 MediSense: Sorry to hear that you're not feeling well. Can you tell me what seems to be the main issue or problem you're experiencing - is it physical, like pain or discomfort somewhere in your body, or more of a general feeling of being unwell?

👤 Patient: My joints hurt, especially in the morning
🤖 MediSense: Morning joint pain can be quite uncomfortable. Is this joint pain localized to just one area, such as your hands, knees, or hips, or is it widespread, affecting multiple joints throughout your body?

👤 Patient: It's been going on for about 2 weeks, mostly my hands and knees
🤖 MediSense: Joint pain in the hands and knees can be a bit of a challenge. Have you noticed any redness, swelling, or warmth around the affected joints, or is it just general aching and stiffness?

📋 MEDISENSE REPORT
Symptoms identified : feeling ill, joint pain
Used for prediction : joint pain

Top predictions:
  osteoporosis                             ███████████ 

In [15]:
# Save final complete artifacts
data['best_model']  = xgb
data['best_name']   = 'XGBoost'
data['GENERIC_SYMPTOMS'] = GENERIC_SYMPTOMS

with open('medisense_preprocessed.pkl', 'wb') as f:
    pickle.dump(data, f)

nlp_data['synonyms'] = SYNONYMS

with open('nlp_artifacts.pkl', 'wb') as f:
    pickle.dump(nlp_data, f)

print("All Phase 5 artifacts saved ✓")
print("\nPhase 5 complete. Ready for Phase 6 — Streamlit Dashboard")

All Phase 5 artifacts saved ✓

Phase 5 complete. Ready for Phase 6 — Streamlit Dashboard
